# 05-5. 데이터 검증

## Goal

필수값·형식·허용값·범위 오류를 구조화한다.


## Setup

`fixtures/05-text-processing/validation-records.jsonl`를 읽는다. 저장소 루트에서 JupyterLab을 실행한다.


In [ ]:
from pathlib import Path
import sys


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError("requirements.txt가 있는 저장소 루트에서 JupyterLab을 실행하세요.")


ROOT = find_project_root()
FIXTURE_DIR = ROOT / "fixtures" / "05-text-processing"

assert sys.version_info >= (3, 10)
assert FIXTURE_DIR.is_dir()

print("Python:", sys.version.split()[0])
print("실습 데이터:", FIXTURE_DIR)


import json

fixture_path = FIXTURE_DIR / "validation-records.jsonl"
raw_lines = fixture_path.read_text(encoding="utf-8").splitlines()
print("전체 JSONL 행:", len(raw_lines))
for line in raw_lines[:2]:
    try:
        print("키:", sorted(json.loads(line)))
    except json.JSONDecodeError:
        print("JSON 문법 오류")


## Steps

JSON 문법 오류와 레코드 검증 오류를 나눈다. `validate_record(record)`는 계정·IPv4/IPv6·인증 결과·위험 점수·요청 경로를 검증하고 `(cleaned, errors)`를 반환한다.


In [ ]:
TODO_DONE = False


def validate_record(record: dict) -> tuple[dict, list[dict]]:
    # 실습 과제: 원본 record를 변경하지 않는다.
    raise NotImplementedError


## Checks

TODO를 구현한 뒤 `TODO_DONE = True`로 바꾸고 공개 경계 검증을 실행한다. 검증이 통과해도 다른 입력이 모두 올바르다는 보장은 아니다.


In [ ]:
if not TODO_DONE:
    print("TODO를 구현한 뒤 TODO_DONE을 True로 바꾸세요.")
else:
    original = {
        "account": " alice_01 ",
        "ip": "2001:0db8:0000::10",
        "result": "success",
        "risk_score": 12,
        "target": "/login?token=do-not-copy",
    }
    before = original.copy()
    cleaned, errors = validate_record(original)
    assert not errors
    assert cleaned["ip"] == "2001:db8::10"
    assert cleaned["result"] == "SUCCESS"
    assert cleaned["target"] == "/login"
    assert original == before
    _, errors = validate_record({})
    assert {error["field"] for error in errors} >= {
        "account", "ip", "result", "risk_score", "target"
    }
    _, errors = validate_record({**original, "risk_score": True})
    assert any(error["field"] == "risk_score" for error in errors)

    fixture_valid, fixture_validation_errors, fixture_json_errors = [], [], []
    for line_number, line in enumerate(raw_lines, start=1):
        try:
            record = json.loads(line)
        except json.JSONDecodeError as error:
            fixture_json_errors.append({"line": line_number, "error": error})
            continue

        before = record.copy()
        cleaned, record_errors = validate_record(record)
        assert record == before
        if record_errors:
            fixture_validation_errors.append({
                "line": line_number,
                "errors": record_errors,
            })
        else:
            fixture_valid.append(cleaned)

    assert len(fixture_valid) == 2
    assert len(fixture_validation_errors) == 2
    assert len(fixture_json_errors) == 1
    assert len(raw_lines) == 5
    assert fixture_valid[0]["target"] == "/login"
    assert fixture_valid[1]["ip"] == "2001:db8::10"
    assert [item["line"] for item in fixture_validation_errors] == [2, 4]
    assert fixture_json_errors[0]["line"] == 5
    print("공개 경계 검증 통과: fixture 정상 2건 / 검증 오류 2건 / JSON 오류 1건")


## Next Steps

파싱 오류와 의미 검증 오류를 서로 다른 코드로 기록한다.
